In [7]:
import pandas as pd
import numpy as np
from data_profiling import ProfileReport

LOAD THE DATA TO CLEAN

In [8]:
data = pd.read_csv(r"C:\Users\1040 G3\Downloads\files\rental_records_raw.csv")

GENERATE A PROFILE REPORT OF THE DATA FRAME

In [9]:
profile = ProfileReport(data, title = 'Profile Report', explorative=True)
profile.to_file(r"C:\Users\1040 G3\Downloads\files\rental_records_profile.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 12/12 [00:00<00:00, 36.43it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

VIEW NUMBER OF COLUMNS AND ROWS

In [10]:
data.shape

(255, 12)

VIEW THE LIST OF COLUMNS

In [11]:
list(data.columns)

['tenant_id',
 'billing_month',
 'tenant_name',
 'phone',
 'unit',
 'monthly_rent',
 'amount_paid',
 'payment_date',
 'payment_method',
 'mpesa_code',
 'reported_balance',
 'status']

OVERVIEW OF THE DATA

In [12]:
data.info

<bound method DataFrame.info of     tenant_id billing_month      tenant_name         phone  unit monthly_rent  \
0        T023       2025-06    naomi wanjiru     761642594   b10     Ksh15000   
1        T005       2025-09    Esther Kamau     0770291817    C5           8k   
2        T038       2025-08     Lucy Wanjiru     071421617    B2        18000   
3        T005       2025-06     ESTHER KAMAU     770291817    C5    KES 8,000   
4        T010       2025-09    Naomi Njoroge  254787490893   A12    KES 8,000   
..        ...           ...              ...           ...   ...          ...   
250      T012       2025-07    Paul  Achieng   0770 855700    C6   KES 10,000   
251      T016       2025-09     lucy njoroge     717507864    a1        12000   
252      T005       2025-07     ESTHER KAMAU     770291817    C5        8,000   
253      T033       2025-10      Ruth Otieno  254745575298   c10     Ksh15000   
254      T030       2025-07    John  Wanjiru    0758718453    b4        8,000

VIEW THE FIRST TEN ROWS

In [25]:
data.head(10)

,tenant_id,billing_month,tenant_name,phone,unit,monthly_rent,amount_paid,payment_date,payment_method,mpesa_code,reported_balance,status
0,T023,2025-06,Naomi Wanjiru,+254761642594,B10,15000.0,7500.0,2025-06-16,M-PESA,JJXWGDL8MX,7500,PARTIAL
1,T005,2025-09,Esther Kamau,+254770291817,C5,8000.0,4000.0,2025-09-04,BANK,BADCODE,4000,PARTIAL
2,T038,2025-08,Lucy Wanjiru,Invalid Phone,B2,18000.0,18000.0,2025-08-18,M-PESA,36PRPLPEA0,0,PAID
3,T005,2025-06,Esther Kamau,+254770291817,C5,8000.0,8000.0,2025-06-19,M-PESA,44C5JRWNA4,0,PAID
4,T010,2025-09,Naomi Njoroge,+254787490893,A12,8000.0,0.0,NaT,M-PESA,NAN,8000,ARREARS
5,T011,2025-11,Mary Kamau,+254748840994,A4,8000.0,8000.0,2025-11-21,BANK,KBTLRB4C55,0,PAID
6,T001,2025-08,Grace Mwangi,+254746913810,A4,10000.0,10000.0,2025-08-06,M-PESA,9K2RQ89U0F,0,PAID
7,T030,2025-11,John Wanjiru,+254758718453,B4,8000.0,8000.0,2025-11-10,M-PESA,DPUSBAC4XS,0,PAID
8,T035,2025-11,Mercy Muthoni,+254779467853,B2,10000.0,5000.0,2025-11-21,BANK,NBHRVXSY0D,5000,PARTIAL
9,T016,2025-10,Lucy Njoroge,+254717507864,A1,12000.0,12000.0,2025-12-10,BANK,BADCODE,0,PAID


CLEAN TENANT NAMES

In [14]:
data['tenant_name'] = (
    data['tenant_name']
    .astype(str)
    .str.title()
    .str.strip()
)

CLEAN PHONE NUMBERS

In [15]:
data['phone'] = (
    data['phone']
    .astype(str)
    .str.replace(' ','')
    .str.strip()
)

def clean_phone(phone):
    if pd.isna(phone):
        return 'Missing Number'
    if phone.startswith(('7','1')):
        phone = '+254' + phone
    elif phone.startswith('0'):
        phone = '+254' + phone[1:]
    elif phone.startswith('254'):
        phone = '+' + phone
    elif phone.startswith('+254'):
        phone = phone
    if len(phone) == 13:
        return phone
    else:
        return 'Invalid Phone'
data['phone'] = data['phone'].map(clean_phone)

CLEAN UNITS

In [16]:
data['unit'] = (
    data['unit']
    .astype(str)
    .str.upper()
    .str.strip()
)

CLEAN MONTHLY RENT

In [17]:
data['monthly_rent'] = (
    data['monthly_rent']
    .astype(str)
    .str.replace(r'(?i)\s*kshs?|kes','',regex=True)
    .str.replace(',','',regex=True)
    .str.strip()
)
data['monthly_rent'] = np.where(data['monthly_rent'].str.endswith('k',na=False),data['monthly_rent'].str.replace('k','').str.strip().astype(float)*1000,data['monthly_rent'])
data['monthly_rent'] = pd.to_numeric(data['monthly_rent'],errors='coerce')

CLEAN AMOUNT PAID

In [18]:
data['amount_paid'] = (
    data['amount_paid']
    .astype(str)
    .str.replace('-','')
    .str.strip()
    .astype(float)
)

CLEAN PAYMENT DATE


In [19]:
data['payment_date'] = pd.to_datetime(data['payment_date'],format='mixed',errors='coerce')

In [20]:
for col in ['payment_method','mpesa_code','status']:
    data[col] = (
        data[col]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r'(?i)bank transfer','BANK',regex=True)
    )

data['payment_method'] = data['payment_method'].replace('MPESA','M-PESA')

Fill the Missing Values in Monthly Rent

In [24]:
data['monthly_rent'] = data['monthly_rent'].fillna(data['tenant_name'].map(data.groupby('tenant_name')['monthly_rent'].first()))


In [30]:
data['payment_method'].unique()
data['mpesa_code'].unique()



array(['JJXWGDL8MX', 'BADCODE', '36PRPLPEA0', '44C5JRWNA4', 'NAN',
       'KBTLRB4C55', '9K2RQ89U0F', 'DPUSBAC4XS', 'NBHRVXSY0D',
       'B5EGPV6RL0', 'UCKEG6TX33', 'T4F8CGWYHE', 'MDQL65XWKY',
       'XK2SW249NP', 'ALZ8P0CZXD', '4XM9967H6Z', 'PCHLD0MEB7',
       'UTF8FY0W4V', 'BG26LLP3SB', '5GWGPX22JP', '88KZSVJN3P',
       '1QRCLQPXKP', 'EXCFHRCX77', 'M73KEZN938', 'MN4SYZJ38X',
       '29ZEC2GUNR', 'W7KZHRJN9N', '7M35HYPKC2', 'QX5D3GP8KP',
       'X01VV80Y6U', 'N3DJ0AT2TT', 'V1YXLQ8JJ7', '5AG44JR7ZK',
       '0FFLB7WYSV', '3D5EN2WAKE', '31VBMC97C7', '61FKHMZ4TJ',
       'W6F1BCRQWQ', 'RVCSL91APB', '09389Y0UKL', 'HGAUK9UZE5',
       'ZMSR2SDXCK', 'FW3YQMDYSW', 'NUDS5YZFC4', 'F5RZ0S6RBE',
       'PNPV9HZ4Y5', 'J12ZEDXB2B', 'F08PBL3F85', '43P84SLKUE',
       'ULCYLWQZMB', 'VV4ZUP12ZC', 'CHJ2W3GDHF', 'SWB2TDSEW4',
       'E6CP0SPPVE', 'C66U4VFEL6', 'PUQHQ6VZ52', '03TF6E7LPW',
       'TNG3PDQVJU', 'LQFN1S4L03', 'P4LH2WLRYT', 'VV9L9YKVZ1',
       'ZZ8TSCBQLJ', '2M0X20MBU3', '0JPYLL1CR9', 'M

In [31]:
for col in ['payment_method','mpesa_code']:
    data[col] = data[col].replace('NAN',np.nan)

In [32]:
data.to_csv(r"C:\Users\1040 G3\OneDrive\Documents\Clean data\clean_rental_records.csv",index=False)